## **Lectura 1: Problema de asignación**


El **problema de asignación** consiste en asignar de manera óptima $n$ agentes a $n$ tareas, de forma que cada agente realice exactamente una tarea y cada tarea sea realizada por exactamente un agente. El criterio de optimización puede ser la minimización del costo (tiempo, distancia, dinero) o la maximización del beneficio.

Algunos ejemplos de aplicación son:
- Asignación de trabajadores a máquinas minimizando el tiempo de producción.
- Asignación de conductores a rutas minimizando el costo de transporte.
- Asignación de proyectos a equipos de trabajo maximizando la productividad.

El problema de asignación es un caso particular del **problema de transporte** con $a_i = b_j = 1$ para todo $i, j$. Por esta razón, el algoritmo símplex de transporte puede aplicarse directamente, aunque su estructura especial permite también el uso del **algoritmo húngaro** (Kuhn, 1955), que resulta más eficiente en la práctica.

### Formulación matemática

Sea $C \in \mathbb{R}^{n \times n}$ la matriz de costos, donde $c_{ij}$ es el costo de asignar el agente $i$ a la tarea $j$.

**Variables de decisión:**

$$x_{ij} = \begin{cases} 1 & \text{si el agente } i \text{ es asignado a la tarea } j \\ 0 & \text{en otro caso} \end{cases}$$

**Variables duales:** $u_i$ asociada a la restricción del agente $i$; $v_j$ asociada a la restricción de la tarea $j$.

**Problema primal:**

$$\min \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij}\, x_{ij}$$

$$\text{s.a.} \quad \sum_{j=1}^{n} x_{ij} = 1 \qquad \forall\, i = 1, \ldots, n \quad \text{(cada agente hace una tarea)}$$

$$\sum_{i=1}^{n} x_{ij} = 1 \qquad \forall\, j = 1, \ldots, n \quad \text{(cada tarea es hecha por un agente)}$$

$$x_{ij} \in \{0, 1\} \qquad \forall\, i, j = 1, \ldots, n$$

Aunque las variables $x_{ij}$ son binarias, la **relajación lineal** (reemplazando $x_{ij} \in \{0,1\}$ por $x_{ij} \geq 0$) produce automáticamente soluciones enteras, gracias a la unimodularidad total de la matriz de restricciones (herencia del problema de transporte).

**Problema dual:**

$$\max \sum_{i=1}^{n} u_i + \sum_{j=1}^{n} v_j$$

$$\text{s.a.} \quad u_i + v_j \leq c_{ij} \qquad \forall\, i, j = 1, \ldots, n$$

$$u_i, v_j \;\text{irrestrictos} \qquad \forall\, i, j$$

La condición de holgura complementaria establece que en el óptimo: si $x_{ij}^* = 1$, entonces $u_i + v_j = c_{ij}$ (la restricción dual es activa). Esta propiedad es la que explotan tanto el algoritmo símplex de transporte como el algoritmo húngaro para calcular las asignaciones óptimas.

**Observación sobre la degeneración:** El problema de asignación de tamaño $n$ tiene $2n$ restricciones de igualdad, por lo que la base del símplex de transporte debería tener $2n - 1$ variables básicas. Sin embargo, una asignación factible solo requiere $n$ variables iguales a 1. La diferencia $n - 1$ corresponde a variables básicas degeneradas con valor cero, lo que genera múltiples bases óptimas asociadas al mismo vértice.

### Sección 1: Aplicación del símplex de transporte al problema de asignación

Dado que el problema de asignación es un caso especial del problema de transporte (con $a_i = b_j = 1$), el algoritmo símplex de transporte puede aplicarse directamente. A continuación se ilustra su aplicación a un ejemplo con 4 agentes y 4 tareas.

Costos de asignación:

![](../Images/ASSIGN_.png)

**Solución básica factible inicial** (obtenida con el método de la esquina noroeste):

![](../Images/ASSIGN_1.png)

Con los potenciales $u_i$ y $v_j$ calculados a partir de los arcos básicos, los costos reducidos de las variables no básicas son:

- $\bar{c}_{11} = 11$, $\bar{c}_{21} = 1$, $\bar{c}_{22} = 9$, $\bar{c}_{23} = 0$
- $\bar{c}_{31} = 8$, $\bar{c}_{32} = 7$, $\bar{c}_{34} = -1$, $\bar{c}_{43} = 4$

El único costo reducido negativo es $\bar{c}_{34} = -1$, por lo que el arco $(3,4)$ entra a la base.

**Iteración del símplex de transporte:**

Se introduce el arco $(3,4)$ al árbol, se identifica el ciclo y se realiza el pivote:

![](../Images/ASSIGN_2.png)

![](../Images/ASSIGN_3.png)

La solución óptima obtenida es **degenerada**: aunque se necesitan $2m - 1 = 7$ variables básicas para mantener la estructura de árbol generador, la asignación factible solo utiliza $m = 4$ variables con valor 1. Las $m - 1 = 3$ variables básicas adicionales tienen valor cero y representan la degeneración estructural característica del problema de asignación.

### Sección 2: Algoritmo húngaro (método de Kuhn-Munkres)

El **algoritmo húngaro** (Kuhn, 1955; Munkres, 1957) es una alternativa al símplex de transporte para resolver el problema de asignación, con complejidad $O(n^3)$. Opera directamente sobre la **matriz de costos reducidos** y utiliza propiedades de la dualidad para garantizar la optimalidad de forma constructiva.

A continuación se ilustra el algoritmo con el siguiente ejemplo:

In [1]:
import pandas as pd

M = ["Maq. "+str(i) for i in range(1,5)]
T = ["Tarea "+str(i) for i in range(1,5)]

# Generar dataframe con costos de asignar cada tarea a cada máquina
asignacion = pd.DataFrame( {T[0]:[14,2,7,2],
                            T[1]:[5,12,8,4],
                            T[2]:[8,6,3,6],
                            T[3]:[7,5,9,10]}, index=M)
asignacion_copia = asignacion.copy()
asignacion

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,14,5,8,7
Maq. 2,2,12,6,5
Maq. 3,7,8,3,9
Maq. 4,2,4,6,10


### Cómo opera el algoritmo húngaro

El algoritmo mantiene en todo momento una **solución dual factible** $(u_i, v_j)$ e intenta construir una asignación primal factible. Los pasos son:

1. **Inicialización:** Calcular una solución dual factible inicial restando el mínimo de cada fila y luego el mínimo de cada columna. Esto equivale a fijar $u_i = \min_j c_{ij}$ y $v_j = \min_i (c_{ij} - u_i)$.

2. **Verificación de "convergencia":** Encontrar el menor número de líneas (horizontales y/o verticales) que cubren todos los ceros de la matriz de costos reducidos. Si el número de líneas es $n$, existe una asignación óptima entre los ceros; si es menor que $n$, se debe ajustar la solución dual.

3. **Ajuste dual:** Encontrar el mínimo $k$ de las celdas no cubiertas, restar $k$ de las celdas no cubiertas y sumar $k$ a las celdas cubiertas dos veces. Esto mantiene la factibilidad dual y genera al menos un nuevo cero.

4. Repetir los pasos 2 y 3 hasta obtener $n$ líneas de cobertura.

En nuestro ejemplo, el problema consiste en encontrar la asignación de mínimo costo entre 4 máquinas y 4 tareas. El algoritmo húngaro opera sobre la matriz de costos reducidos, buscando en cada iteración un nuevo cero que permita avanzar hacia una asignación completa.

In [2]:
# Encontrar mínimo de cada fila
asignacion["min_cada_fila"] = asignacion.min(axis=1)
asignacion

,Tarea 1,Tarea 2,Tarea 3,Tarea 4,min_cada_fila
Maq. 1,14,5,8,7,5
Maq. 2,2,12,6,5,2
Maq. 3,7,8,3,9,3
Maq. 4,2,4,6,10,2


In [3]:
# Restar a cada fila su mínimo (por practicidad hago la operación por columnas, pero la operación es en filas)
for i in range(4):
    asignacion.iloc[:,i] = asignacion.iloc[:,i].subtract(asignacion["min_cada_fila"])
asignacion

,Tarea 1,Tarea 2,Tarea 3,Tarea 4,min_cada_fila
Maq. 1,9,0,3,2,5
Maq. 2,0,10,4,3,2
Maq. 3,4,5,0,6,3
Maq. 4,0,2,4,8,2


In [4]:
import numpy as np

# encontrar mínimo de cada (nueva) columna
min_cols = asignacion.min()

# registrarlo
nueva_fila = pd.DataFrame( dict(zip(T+["min_cada_fila"], min_cols)), index=["min_cada_col"] )
nueva_fila.iloc[0,4] = np.nan

asignacion = asignacion.append(nueva_fila)
asignacion

,Tarea 1,Tarea 2,Tarea 3,Tarea 4,min_cada_fila
Maq. 1,9,0,3,2,5.0
Maq. 2,0,10,4,3,2.0
Maq. 3,4,5,0,6,3.0
Maq. 4,0,2,4,8,2.0
min_cada_col,0,0,0,2,NaN


In [5]:
# restar a cada columna su mínimo
for i in range(4):
    asignacion.iloc[0:3,i] = asignacion.iloc[0:3,i] - asignacion.iloc[4,i]

# eliminar columnas agregadas antes
asignacion.drop("min_cada_fila", inplace=True, axis=1)
asignacion.drop("min_cada_col", inplace=True)

asignacion

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,9,0,3,0
Maq. 2,0,10,4,1
Maq. 3,4,5,0,4
Maq. 4,0,2,4,8


In [6]:
import resaltar

# Tachar ceros con MÍNIMA cantidad de líneas horizontales y verticales
asignacion.style.apply(resaltar.tabla, filas=[0,2], cols=[0], col_names=asignacion.columns)

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,9,0,3,0
Maq. 2,0,10,4,1
Maq. 3,4,5,0,4
Maq. 4,0,2,4,8


In [7]:
# Dado que hay 3 líneas tachadas, y se deben hacer 4 asignaciones, NO hay convergencia aun...

# Encontrar mínimo de celdas no cubiertas y llamarlo k
k = min( asignacion.iloc[[1,3],[1,2,3]].min() )
print("Mínimo no cubierto es k="+str(k))

# Restar k de celdas no cubiertas (encontradas por inspección visual)
asignacion.iloc[[1,3],[1,2,3]] = asignacion.iloc[[1,3],[1,2,3]].apply(lambda x: x-k)

# Sumar k a celdas cubiertas dos veces (encontradas por inspección visual)
asignacion.iloc[0,0] = asignacion.iloc[0,0] + k
asignacion.iloc[2,0] = asignacion.iloc[2,0] + k

asignacion.style.apply(resaltar.celdas, cells=[(0,0),(2,0),(1,3)], col_names=asignacion.columns)

Mínimo no cubierto es k=1


,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,10,0,3,0
Maq. 2,0,9,3,0
Maq. 3,5,5,0,4
Maq. 4,0,1,3,7


Después del ajuste dual con $k = 1$, los cambios en la tabla son:

- Los costos reducidos de la Maq. 2 en las Tareas 1, 2, 3 pasan de $10, 4, 1$ a $9, 3, 0$.
- Los costos reducidos de la Maq. 4 en las Tareas 1, 2, 3 pasan de $2, 4, 8$ a $1, 3, 7$.
- Las celdas en los cruces de filas y columnas cubiertas dos veces se incrementan en $k$.

Se genera un nuevo cero en la posición (Maq. 2, Tarea 3).

In [8]:
# Tachar ceros con MÍNIMA cantidad de líneas horizontales y verticales
asignacion.style.apply(resaltar.tabla, filas=[0,2], cols=[0,3], col_names=asignacion.columns)

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,10,0,3,0
Maq. 2,0,9,3,0
Maq. 3,5,5,0,4
Maq. 4,0,1,3,7


Con 4 líneas de cobertura para una instancia de tamaño $n = 4$, se ha alcanzado la condición de convergencia. Ahora es posible seleccionar una asignación factible entre los ceros de la tabla de costos reducidos (sin asignar una tarea a más de una máquina ni una máquina a más de una tarea).

In [9]:
asignacion.style.apply(resaltar.celdas, cells=[(0,1),(1,3),(2,2),(3,0)], col_names=asignacion.columns)

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,10,0,3,0
Maq. 2,0,9,3,0
Maq. 3,5,5,0,4
Maq. 4,0,1,3,7


Se ignoran los ceros adicionales que generarían dobles asignaciones. La asignación seleccionada es **óptima**. ¿Por qué?

In [10]:
# Revisar los costos de esa asignación sobre los costos originales del problema
asignacion_copia.style.apply(resaltar.celdas, cells=[(0,1),(1,3),(2,2),(3,0)], col_names=asignacion.columns)

,Tarea 1,Tarea 2,Tarea 3,Tarea 4
Maq. 1,14,5,8,7
Maq. 2,2,12,6,5
Maq. 3,7,8,3,9
Maq. 4,2,4,6,10


El costo total de la asignación es $\$15$. Puede verificarse que ninguna asignación alternativa produce un costo menor: esto es consecuencia directa del teorema de dualidad fuerte.

### Sección 3: Conexión entre el algoritmo húngaro y la dualidad

El problema de asignación puede escribirse como:

$$\min \sum_{i \in \mathcal{O}} \sum_{j \in \mathcal{D}} c_{ij}\, x_{ij}$$

$$\sum_{j \in \mathcal{D}} x_{ij} = 1 \qquad \forall\, i \in \mathcal{O}$$

$$\sum_{i \in \mathcal{O}} x_{ij} = 1 \qquad \forall\, j \in \mathcal{D}$$

$$x_{ij} \geq 0 \qquad \forall\, i \in \mathcal{O},\, j \in \mathcal{D}$$

El problema dual (relajando la integralidad) es:

$$\max \sum_{i \in \mathcal{O}} u_i + \sum_{j \in \mathcal{D}} v_j$$

$$\text{s.a.} \quad u_i + v_j \leq c_{ij} \qquad \forall\, i \in \mathcal{O},\, j \in \mathcal{D}$$

$$u_i, v_j \;\text{irrestrictos}$$

Por **holgura complementaria**, una asignación $x^*$ es óptima si y solo si existe una solución dual $(u^*, v^*)$ tal que $u_i^* + v_j^* = c_{ij}$ siempre que $x_{ij}^* = 1$. El algoritmo húngaro construye exactamente este par primal-dual.

In [11]:
min_filas = list(asignacion_copia.min(axis=1))
min_cols = [0,0,0,2] # mínimos sobre columna, pero después de restar, ver paso XX

duales = pd.DataFrame( {"u(i)":min_filas, "v(j)":min_cols}, index=[1,2,3,4] )
duales

,u(i),v(j)
1,5,0
2,2,0
3,3,0
4,2,2


In [12]:
def actualizar_costos_red(duales):
    # Conjunto de arcos
    arcos = [(i,j) for i in range(1,5) for j in range(1,5)]
    
    # Costos reducidos
    cred = ["c("+str(i)+","+str(j)+")  -  u("+str(i)+")  -  v("+str(j)+")" for i in range(1,5) for j in range(1,5)]

    c = [asignacion_copia.iloc[i-1,j-1] for i in range(1,5) for j in range(1,5)]
    U = [duales.iloc[i-1,0] for i in range(1,5) for j in range(1,5)]
    V = [duales.iloc[j-1,1] for i in range(1,5) for j in range(1,5)]
    v_cred = [c[i]-U[i]-V[i] for i in range(len(c))]

    asignable = [i==0 for i in v_cred]    

    DF = pd.DataFrame({"(i,j)":arcos, 
                       "Valor costo c(i,j)":c, 
                       "Holgura Dual / C.Redu.Primal":cred,
                       "Valor u(i)":U,
                       "Valor v(j)":V,
                       "Result. 'costo red'":v_cred,
                       "Asignable?":asignable})
    DF.set_index("(i,j)", inplace=True)
    return DF

DF = actualizar_costos_red(duales)
colorear = [i for i in range(len(DF["Asignable?"])) if list(DF["Asignable?"])[i]]

DF.style.apply(resaltar.tabla, filas=colorear, cols=[], col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",9,14,5,0
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",True,"c(2,1) - u(2) - v(1)",0,2,2,0
"(2, 2)",False,"c(2,2) - u(2) - v(2)",10,12,2,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",4,6,2,0
"(2, 4)",False,"c(2,4) - u(2) - v(4)",1,5,2,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",4,7,3,0


Con la solución dual actual no es posible construir una asignación primal factible: no existe un conjunto de $n$ ceros en la matriz de costos reducidos que cubra todas las filas y columnas simultáneamente.

Para avanzar es necesario ajustar las variables duales $u_i$ y $v_j$, generando un nuevo cero (haciendo activa una nueva restricción dual) **sin perder la factibilidad dual** (sin generar costos reducidos negativos).

El ajuste consiste en cambiar los valores de algunas variables duales respetando las siguientes condiciones:
- No se pierden los ceros existentes que participan en la asignación actual.
- No aparecen costos reducidos negativos.

In [13]:
duales.iloc[1,0] = 3

DF = actualizar_costos_red(duales)
colorear = [(i,j) for i in range(4,8) for j in [2,4]]

DF.style.apply(resaltar.celdas, cells=colorear, col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",9,14,5,0
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",False,"c(2,1) - u(2) - v(1)",-1,2,3,0
"(2, 2)",False,"c(2,2) - u(2) - v(2)",9,12,3,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",3,6,3,0
"(2, 4)",True,"c(2,4) - u(2) - v(4)",0,5,3,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",4,7,3,0


Al modificar $u_2$ de 2 a 3, los costos reducidos de todas las celdas que dependen de $u_2$ cambian:

- Se obtiene el cero deseado en la posición $(2,4)$.
- En la posición $(2,1)$ aparece un valor **negativo**, lo que viola la factibilidad dual.

Para corregir esto, se debe ajustar $v_1$ de modo que el costo reducido en $(2,1)$ vuelva a ser no negativo.

In [14]:
duales.iloc[0,1] = -1

DF = actualizar_costos_red(duales)
colorear = [(i,j) for i in [0,4,8,12] for j in [2,5]]

DF.style.apply(resaltar.celdas, cells=colorear, col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",10,14,5,-1
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",True,"c(2,1) - u(2) - v(1)",0,2,3,-1
"(2, 2)",False,"c(2,2) - u(2) - v(2)",9,12,3,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",3,6,3,0
"(2, 4)",True,"c(2,4) - u(2) - v(4)",0,5,3,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",5,7,3,-1


Al modificar $v_1$ de 0 a $-1$, los costos reducidos de todas las celdas de la columna 1 cambian:

- Se recupera la no negatividad en $(2,1)$.
- Se pierde el cero en $(4,1)$: la posición $(4,1)$ queda con costo reducido positivo.

Para recuperar ese cero es necesario ajustar $u_4$.

In [15]:
duales.iloc[3,0] = 3

DF = actualizar_costos_red(duales)
colorear = [(i,j) for i in range(12,16) for j in [2,4]]

DF.style.apply(resaltar.celdas, cells=colorear, col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",10,14,5,-1
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",True,"c(2,1) - u(2) - v(1)",0,2,3,-1
"(2, 2)",False,"c(2,2) - u(2) - v(2)",9,12,3,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",3,6,3,0
"(2, 4)",True,"c(2,4) - u(2) - v(4)",0,5,3,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",5,7,3,-1


Tras los ajustes sucesivos de las variables duales, se ha recuperado la factibilidad dual sin generar costos reducidos negativos. La solución dual modificada genera un nuevo cero en la tabla y permite ahora encontrar una asignación factible.

In [16]:
colorear = [i for i in range(len(DF["Asignable?"])) if list(DF["Asignable?"])[i]]

DF.style.apply(resaltar.tabla, filas=colorear, cols=[], col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",10,14,5,-1
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",True,"c(2,1) - u(2) - v(1)",0,2,3,-1
"(2, 2)",False,"c(2,2) - u(2) - v(2)",9,12,3,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",3,6,3,0
"(2, 4)",True,"c(2,4) - u(2) - v(4)",0,5,3,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",5,7,3,-1


Con la nueva configuración de ceros existe al menos una asignación factible: se puede seleccionar $n = 4$ ceros tales que no compartan fila ni columna.

In [17]:
colorear = [1,7,10,12]

DF.style.apply(resaltar.tabla, filas=colorear, cols=[], col_names=DF.columns)

,Asignable?,Holgura Dual / C.Redu.Primal,Result. 'costo red',"Valor costo c(i,j)",Valor u(i),Valor v(j)
"(i,j)",,,,,,
"(1, 1)",False,"c(1,1) - u(1) - v(1)",10,14,5,-1
"(1, 2)",True,"c(1,2) - u(1) - v(2)",0,5,5,0
"(1, 3)",False,"c(1,3) - u(1) - v(3)",3,8,5,0
"(1, 4)",True,"c(1,4) - u(1) - v(4)",0,7,5,2
"(2, 1)",True,"c(2,1) - u(2) - v(1)",0,2,3,-1
"(2, 2)",False,"c(2,2) - u(2) - v(2)",9,12,3,0
"(2, 3)",False,"c(2,3) - u(2) - v(3)",3,6,3,0
"(2, 4)",True,"c(2,4) - u(2) - v(4)",0,5,3,2
"(3, 1)",False,"c(3,1) - u(3) - v(1)",5,7,3,-1


La asignación encontrada coincide con la obtenida mediante las reglas del algoritmo húngaro. Los ceros no utilizados corresponden a la **degeneración estructural** del problema: la base del símplex de transporte necesita $2n - 1 = 7$ variables básicas, pero solo $n = 4$ de ellas toman valor 1 en la asignación óptima.

**Conexión formal entre el algoritmo húngaro y la dualidad:**

- Al restar los mínimos de fila y columna en la inicialización, se calculan los **costos reducidos** del primal, equivalentes a las holguras del dual, asignando valores factibles a las variables duales (que garantizan la no negatividad de los costos reducidos).
- Esta solución dual inicial es **factible** para el dual, pero la asignación primal asociada generalmente **no es factible** (no cubre todas las filas y columnas con ceros independientes).
- Cada ajuste de las variables duales ($u_i$, $v_j$) corresponde a un **paso del símplex dual**: se busca mejorar la asignación primal sin perder la factibilidad dual.
- La convergencia se produce cuando existe una asignación primal factible en los ceros de la tabla de costos reducidos; en ese punto, tanto el primal como el dual son factibles y óptimos por el teorema de dualidad fuerte.

### Glosario de conceptos de programación lineal

Los siguientes conceptos de programación lineal son relevantes para comprender el funcionamiento del algoritmo húngaro y su relación con la dualidad:

- **Holgura complementaria:** Si una restricción primal se cumple con igualdad (holgura cero), la variable dual correspondiente puede ser positiva. Si la restricción es inactiva (holgura positiva), la variable dual debe ser cero. Esto se formaliza como $x_j \cdot \bar{c}_j = 0$ para toda variable $j$.
- **Factibilidad dual:** Una solución dual $(u, v)$ es factible si satisface $u_i + v_j \leq c_{ij}$ para todo $(i,j)$. Equivalentemente, todos los costos reducidos $\bar{c}_{ij} = c_{ij} - u_i - v_j$ son no negativos.
- **Dualidad fuerte:** Si el primal y el dual son factibles, sus valores óptimos coinciden: $z^* = w^*$. En el problema de asignación, esto garantiza que la solución encontrada por el algoritmo húngaro es globalmente óptima.
- **Degeneración:** Ocurre cuando una variable básica tiene valor cero (en el caso primal) o cuando una restricción inactiva tiene variable dual positiva (en el caso dual). En el problema de asignación, la degeneración es estructural: siempre existen $n - 1$ variables básicas con valor cero.

### Ejercicio

Resuelva mediante el algoritmo húngaro el siguiente problema de asignación: el entrenador de la selección colombiana de natación debe determinar qué nadadores participarán en una carrera de relevo 4×100 metros, minimizando el tiempo total.

![](../Images/ASSIGN_4.png)

Indique en cada iteración de la inicialización los ceros generados, la cobertura mínima y, si es necesario, el ajuste dual. Verifique la optimalidad mediante las condiciones de holgura complementaria.

---

## Referencias

1. Bazaraa, M. S., Jarvis, J. J. y Sherali, H. D. (1990). *Linear Programming and Network Flows* (2.ª ed.). Wiley. (pp. 527–554) — Problema de asignación como caso especial del problema de transporte.
2. Ahuja, R. K., Magnanti, T. L. y Orlin, J. B. (1993). *Network Flows: Theory, Algorithms, and Applications*. Prentice-Hall. (Cap. 11)
3. Kuhn, H. W. (1955). The Hungarian method for the assignment problem. *Naval Research Logistics Quarterly*, *2*(1–2), 83–97. https://doi.org/10.1002/nav.3800020109 — Algoritmo húngaro (algoritmo de Kuhn-Munkres).
4. Munkres, J. (1957). Algorithms for the assignment and transportation problems. *Journal of the Society for Industrial and Applied Mathematics*, *5*(1), 32–38. — Análisis de complejidad del algoritmo húngaro.
